# AgentSynth — RL with verified rewards (GRPO, Colab)

GRPO needs a reward signal, and "did the tool call actually work?" is a better one
than any LLM's opinion. This notebook trains a small **base** model (zero
function-calling ability) to call the tools of a REST API — defined by nothing but
its **OpenAPI spec** — with rewards from AgentSynth's verification stack: parse,
real tool, valid args, **real execution**. Runs on a **free Colab T4**.

Swap the demo spec for your own API and the same loop applies.

First: **Runtime → Change runtime type → T4 GPU**.

## 1. Install

In [ ]:
%pip install -q "agentsynth-ai>=0.4.0" unsloth
# Bleeding edge? %pip install -q "agentsynth-ai @ git+https://github.com/agentsynth/agentsynth" unsloth

## 2. An API to learn — from its OpenAPI spec

A tiny orders API on loopback so the notebook is self-contained. **This is the cell
you replace** to train against your own service: point `RestEnvironment` at your
spec URL (and a staging base_url).

In [ ]:
import json, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

from agentsynth.environments import RestEnvironment

SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Order API", "version": "1"},
    "paths": {
        "/orders/{order_id}": {
            "get": {
                "operationId": "get_order",
                "summary": "Fetch one order with its status and total.",
                "parameters": [{"name": "order_id", "in": "path", "required": True,
                                "schema": {"type": "integer"}}],
            }
        },
        "/orders": {
            "get": {
                "operationId": "list_orders",
                "summary": "List recent orders, newest first.",
                "parameters": [{"name": "limit", "in": "query",
                                "schema": {"type": "integer"}}],
            }
        },
    },
}

_ORDERS = {n: {"id": n, "status": "shipped", "total": round(19.99 * n, 2)} for n in range(1, 9)}

class _Api(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass
    def do_GET(self):
        path = self.path.split("?")[0]
        if path == "/orders":
            payload = list(_ORDERS.values())
        elif path.startswith("/orders/"):
            try:
                payload = _ORDERS.get(int(path.rsplit("/", 1)[-1]), {"error": "no such order"})
            except ValueError:
                payload = {"error": "bad id"}
        else:
            payload = {"error": "not found"}
        body = json.dumps(payload).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.end_headers(); self.wfile.write(body)

server = ThreadingHTTPServer(("127.0.0.1", 0), _Api)
threading.Thread(target=server.serve_forever, daemon=True).start()
host, port = server.server_address

env = RestEnvironment(SPEC, base_url=f"http://{host}:{port}")
print("tools from the spec:", env.tool_names())

## 3. Prompts + the verified reward

The reward function scores each completion on parse / real tool / required args /
**actual execution against the API** — deterministic, no LLM judge needed.

In [ ]:
from datasets import Dataset

from agentsynth import make_reward_fn

TOOLS_JSON = json.dumps([
    {"name": t.name, "description": t.description, "parameters": t.parameters}
    for t in env.tools()
])
TEMPLATE = ("You can call exactly one tool to help the user.\n"
            "Tools (JSON): " + TOOLS_JSON + "\n\n"
            "User: {query}\n"
            'Respond with ONLY a JSON object: {{"tool": "<tool name>", "args": {{<arguments>}}}}')

queries = [f"What is the status of order {n}?" for n in range(1, 9)]
queries += [f"Show me the {n} most recent orders." for n in (3, 5, 10)]
queries += ["Give me the total amount of order 4.", "Which orders shipped recently?",
            "Look up order 7.", "How much was order 2?", "List today's orders."]

ds = Dataset.from_list([{"prompt": TEMPLATE.format(query=q)} for q in queries])
reward_fn = make_reward_fn(environment=env)
print(len(ds), "prompts | reward components: parse, tool, args, execution")

## 4. Load the base model and measure the reward *before*

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-1B", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0.0)

def sample_completions(n=8, max_new_tokens=48):
    FastLanguageModel.for_inference(model)
    outs = []
    for row in ds.select(range(n)):
        ids = tokenizer(row["prompt"], return_tensors="pt").input_ids.to(model.device)
        out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        outs.append(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    return outs

before_samples = sample_completions()
before_rewards = reward_fn(prompts=None, completions=before_samples)
print("mean reward BEFORE:", round(sum(before_rewards) / len(before_rewards), 3))
print("sample:", repr(before_samples[0][:90]))

## 5. GRPO — the verified reward is the training signal

A few minutes on a T4. Watch `rewards/agentsynth_verified_reward/mean` climb in the
logs — that's the model learning to emit calls that *actually run*.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    train_dataset=ds,
    processing_class=tokenizer,
    args=GRPOConfig(
        output_dir="out/grpo",
        max_steps=30,
        num_generations=4,
        per_device_train_batch_size=4,
        max_completion_length=48,
        learning_rate=5e-6,
        logging_steps=5,
        report_to=[],
    ),
)
trainer.train()

## 6. Measure the reward *after*

In [ ]:
after_samples = sample_completions()
after_rewards = reward_fn(prompts=None, completions=after_samples)

print(f"mean verified reward: {sum(before_rewards)/len(before_rewards):.3f}  ->  "
      f"{sum(after_rewards)/len(after_rewards):.3f}")
print("\nAFTER sample:", repr(after_samples[0][:120]))

## Where to go next

- **Your API**: replace cell 2 with `RestEnvironment("https://your.host/openapi.json")`
  (use a staging server, or `methods=("get",)` for reads only).
- **Any MCP server or a browser** works the same way — every AgentSynth environment
  is a reward source.
- **Step-by-step episodes**: `agentsynth.AgentGym` runs gym-style multi-turn episodes
  whose terminal reward adds the full verification stack + the judge, and
  `agentsynth.rl.to_openenv` serves them over the OpenEnv standard.
- Docs: https://agentsynth.github.io/agentsynth · Repo: https://github.com/agentsynth/agentsynth